<a href="https://colab.research.google.com/github/anushikps625-create/Mechanism-synthesis/blob/main/machine_design.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import Wedge
from scipy.optimize import fsolve
from IPython.display import display, HTML
import matplotlib.transforms as transforms
import matplotlib.patches as patches
import sys
from matplotlib.animation import FuncAnimation
import matplotlib.patheffects as pe


In [ ]:


class KinematicDynamicAnalyzer:
    def __init__(self, synthesizer, mass=0.5, inertia=0.001, omega_crank=10.0):
        """
        Initializes the analyzer.
        mass: kg, inertia: kg-m^2, omega_crank: rad/s
        """
        self.synth = synthesizer
        self.m = mass
        self.I = inertia
        self.omega_c = omega_crank
        self.kinematics = {}
        self.dynamics = {}

    def perform_kinematic_analysis(self):
        """
        Calculates velocities and accelerations using discrete finite differences
        across the precision points.
        """
        # QRM Forward stroke spans 240 degrees (4.188 rad) for a TR of 2
        t_forward = math.radians(240.0) / self.omega_c

        # Approximate time at B based on proportion of vertical displacement
        drop_b = self.synth.drop_b
        drop_c = self.synth.drop_c
        t_A = 0.0
        t_B = t_forward * (drop_b / drop_c)
        t_C = t_forward

        dt_AB = t_B - t_A
        dt_BC = t_C - t_B

        # Linear Velocities (m/s)
        v_y_AB = (-drop_b - 0) / dt_AB / 1000.0
        v_y_BC = (-drop_c - (-drop_b)) / dt_BC / 1000.0

        # Angular Velocities (rad/s) fetched dynamically
        theta_A = self.synth.states[0]['theta_deg']
        theta_B = self.synth.states[1]['theta_deg']
        theta_C = self.synth.states[2]['theta_deg']

        omega_AB = math.radians(theta_B - theta_A) / dt_AB
        omega_BC = math.radians(theta_C - theta_B) / dt_BC

        # Accelerations at State B using central difference
        dt_avg = (dt_AB + dt_BC) / 2.0
        a_y_B = (v_y_BC - v_y_AB) / dt_avg # m/s^2
        alpha_B = (omega_BC - omega_AB) / dt_avg # rad/s^2

        self.kinematics = {
            't_forward': t_forward,
            'v_y_AB': v_y_AB, 'v_y_BC': v_y_BC,
            'omega_AB': omega_AB, 'omega_BC': omega_BC,
            'a_y_B': a_y_B, 'alpha_B': alpha_B
        }
        return self.kinematics

    def perform_dynamic_analysis(self):
        """
        Uses Newton-Euler equations to calculate inertial forces and torques at State B.
        """
        if not self.kinematics:
            self.perform_kinematic_analysis()

        a_y = self.kinematics['a_y_B']
        alpha = self.kinematics['alpha_B']

        # F = ma (Inertial force required in Y direction)
        g = 9.81
        F_inertial_y = self.m * a_y

        # Total force includes gravitational assistance (since motion is downward)
        F_total_y = F_inertial_y + (self.m * g)

        # T = I*alpha (Torque required to angularly accelerate the object)
        Torque_inertial = self.I * alpha

        self.dynamics = {
            'F_inertial_y': F_inertial_y,
            'F_total_y': F_total_y,
            'Torque_inertial': Torque_inertial
        }
        return self.dynamics

    def display_analysis(self):
        print("\n--- Kinematic Analysis Results ---")
        print(f"Assumed Crank Speed:       {self.omega_c} rad/s")
        print(f"Forward Stroke Time:       {self.kinematics['t_forward']:.4f} s")
        print(f"Avg Linear Vel A->B:       {self.kinematics['v_y_AB']:.4f} m/s")
        print(f"Avg Linear Vel B->C:       {self.kinematics['v_y_BC']:.4f} m/s")
        print(f"Avg Angular Vel A->B:      {self.kinematics['omega_AB']:.4f} rad/s")
        print(f"Avg Angular Vel B->C:      {self.kinematics['omega_BC']:.4f} rad/s")
        print(f"Instant Linear Accel @ B:  {self.kinematics['a_y_B']:.4f} m/s^2")
        print(f"Instant Angular Accel @ B: {self.kinematics['alpha_B']:.4f} rad/s^2")

        print("\n--- Dynamic Analysis Results ---")
        print(f"Assumed Mass:              {self.m} kg")
        print(f"Assumed Inertia (Izz):     {self.I} kg-m^2")
        print(f"Inertial Force (Y) @ B:    {self.dynamics['F_inertial_y']:.4f} N")
        print(f"Total Y Force (w/ grav):   {self.dynamics['F_total_y']:.4f} N")
        print(f"Inertial Torque @ B:       {self.dynamics['Torque_inertial']:.4f} N-m")

In [ ]:
class MechanismSynthesizer:
    def __init__(self, radius, drop_b, drop_c):
        self.radius = radius
        self.drop_b = drop_b
        self.drop_c = drop_c
        self.states = []
        self.qrm_links = {}
        self.coupler_links = {}
        #coupler_bracket.set_data([], [])

    def define_precision_points(self):
        self.states = [
            {'id': 'A', 'y': 0.0, 'theta_deg': 0.0},
            {'id': 'B', 'y': -self.drop_b, 'theta_deg': -45.0},
            {'id': 'C', 'y': -self.drop_c, 'theta_deg': -90.0}
        ]
        return self.states

    def synthesize_quick_return(self, target_tr=2.0):
        alpha_rad = math.radians(360.0 / (target_tr + 1.0))
        crank_r = 10.0
        ground_d = crank_r / math.cos(alpha_rad / 2.0)
        target_stroke = self.drop_c
        lever_l = target_stroke / (2.0 * (crank_r / ground_d))

        self.qrm_links = {
            'crank_radius': crank_r,
            'ground_distance': ground_d,
            'lever_length': lever_l,
            'target_tr': target_tr,
            'actual_stroke': target_stroke
        }
        return self.qrm_links

    # --- NEW FUNCTION FOR PHASE 3 ---
    def synthesize_rotational_coupler(self, u=20.0, v=0.0):
        """
        Calculates the grounded pin joint and link length to enforce rotation.
        (u, v) are the local coordinates of the pin on the moving object.
        """
        pin_absolute_positions = []

        for state in self.states:
            y = state['y']
            theta_rad = math.radians(state['theta_deg'])

            # 2D Rotation matrix applied to local pin coordinates, plus vertical translation
            Xp = u * math.cos(theta_rad) - v * math.sin(theta_rad)
            Yp = y + u * math.sin(theta_rad) + v * math.cos(theta_rad)
            pin_absolute_positions.append((Xp, Yp))

        try:
            ground_pivot, link_length = find_circumcenter(*pin_absolute_positions)

            self.coupler_links = {
                'local_pin_u': u,
                'local_pin_v': v,
                'ground_pivot_X': ground_pivot[0],
                'ground_pivot_Y': ground_pivot[1],
                'coupler_length': link_length,
                'pin_positions': pin_absolute_positions
            }
            return self.coupler_links
        except ValueError as e:
            print(f"Coupler synthesis failed: {e}")
            return None

    def display_coupler_specs(self):
        """Prints the synthesized geometry for the rotational coupler."""
        if not self.coupler_links:
            return

        print(f"\n{'Coupler Component':<25} | {'Value':<20}")
        print("-" * 48)
        print(f"{'Local Pin on Object (u, v)':<25} | ({self.coupler_links['local_pin_u']}, {self.coupler_links['local_pin_v']}) mm")
        print(f"{'Fixed Ground Pivot (X, Y)':<25} | ({self.coupler_links['ground_pivot_X']:.2f}, {self.coupler_links['ground_pivot_Y']:.2f}) mm")
        print(f"{'Coupler Link Length (L)':<25} | {self.coupler_links['coupler_length']:.2f} mm")
        print("-" * 48)
        print("Absolute Pin Positions at A, B, C:")
        for idx, pos in enumerate(['A', 'B', 'C']):
            x, y = self.coupler_links['pin_positions'][idx]
            print(f"  State {pos}: ({x:.2f}, {y:.2f})")

In [ ]:
def run_dynamic_mechanism(radius_mm, drop_b_mm, drop_c_mm, omega=10.0):
    print(f"\n--- Calculating Perfect Synchronized Mechanism ---")

    # 1. Pin remains safely on the far left edge of the circle
    pin_u = -radius_mm
    pin_v = 0.0

    # 2. The grounded pivot used by the rotational coupler
    Xg = -drop_c_mm / 2.0
    Yg = -drop_c_mm / 2.0

    # 3. Exact rigid coupler length
    L_coupler = math.sqrt((pin_u - Xg) ** 2 + (pin_v - Yg) ** 2)

    print(f"Perfect Sync Pivot Grounded at: X = {Xg:.3f}, Y = {Yg:.3f}")
    print(f"Required Coupler Length: {L_coupler:.3f} mm")

    print("\n--- Generating Animation ---")

    animator = FullMechanismAnimator(
        drop_c=drop_c_mm,
        u=pin_u,
        v=pin_v,
        Xg=Xg,
        Yg=Yg,
        L_coupler=L_coupler,
        radius=radius_mm,
        omega=omega
    )

    anim = animator.create_animation()
    display(HTML(anim.to_jshtml()))


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as pe
import matplotlib.animation as animation


def find_circumcenter(p1, p2, p3):
    """
    Calculates the center and radius of a circle passing through three 2D points.
    Used to find the grounded pin joint location and the coupler link length.
    """
    x1, y1 = p1
    x2, y2 = p2
    x3, y3 = p3

    D = 2 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))

    if abs(D) < 1e-9:
        raise ValueError(
            "Points are collinear. No unique rotational coupler exists for these pin coordinates."
        )

    Ux = (
        ((x1**2 + y1**2) * (y2 - y3) +
         (x2**2 + y2**2) * (y3 - y1) +
         (x3**2 + y3**2) * (y1 - y2)) / D
    )
    Uy = (
        ((x1**2 + y1**2) * (x3 - x2) +
         (x2**2 + y2**2) * (x1 - x3) +
         (x3**2 + y3**2) * (x2 - x1)) / D
    )

    radius = math.hypot(Ux - x1, Uy - y1)
    return (Ux, Uy), radius


def three_position_synthesis(radius, drop_b, drop_c):
    def get_circumcenter(Ax, Ay, Bx, By, Cx, Cy):
        D = 2 * (Ax * (By - Cy) + Bx * (Cy - Ay) + Cx * (Ay - By))
        if abs(D) < 1e-6:
            return None, None
        Ux = (
            ((Ax**2 + Ay**2) * (By - Cy) +
             (Bx**2 + By**2) * (Cy - Ay) +
             (Cx**2 + Cy**2) * (Ay - By)) / D
        )
        Uy = (
            ((Ax**2 + Ay**2) * (Cx - Bx) +
             (Bx**2 + By**2) * (Ax - Cx) +
             (Cx**2 + Cy**2) * (Bx - Ax)) / D
        )
        return Ux, Uy

    best_u, best_v = 0, 0
    best_Xg, best_Yg, best_L = 0, 0, 0
    min_error = float('inf')

    sweep = -math.pi / 2
    theta_B = sweep * (drop_b / drop_c)
    theta_C = sweep

    steps = 60
    for u in np.linspace(-radius, radius, steps):
        for v in np.linspace(-radius, radius, steps):
            if u > 0:
                continue
            if u**2 + v**2 > (radius * 0.95)**2:
                continue
            if u**2 + v**2 < (radius * 0.3)**2:
                continue

            Ax, Ay = u, v
            Bx = u * math.cos(theta_B) - v * math.sin(theta_B)
            By = -drop_b + u * math.sin(theta_B) + v * math.cos(theta_B)
            Cx = u * math.cos(theta_C) - v * math.sin(theta_C)
            Cy = -drop_c + u * math.sin(theta_C) + v * math.cos(theta_C)

            Xg, Yg = get_circumcenter(Ax, Ay, Bx, By, Cx, Cy)
            if Xg is None:
                continue
            if Xg > -radius * 0.1:
                continue

            L = math.hypot(Ax - Xg, Ay - Yg)
            if L < radius * 0.2 or L > radius * 15:
                continue

            valid_kinematics = True
            R1 = math.hypot(u, v)
            for y_test in np.linspace(-drop_c - 1.0, 1.0, 20):
                dist = math.hypot(-Xg, y_test - Yg)
                margin = 0.5
                if dist > (R1 + L - margin) or dist < abs(R1 - L) + margin:
                    valid_kinematics = False
                    break

            if not valid_kinematics:
                continue

            error = L + abs(Yg)
            if error < min_error:
                min_error = error
                best_u, best_v = u, v
                best_Xg, best_Yg, best_L = Xg, Yg, L

    if min_error == float('inf'):
        raise ValueError(
            "Geometry constraints are mathematically impossible for a perfect 90-degree sweep with these inputs."
        )

    print("Optimal geometry found! Locked to exact 90° sweep.")
    return best_u, best_v, best_Xg, best_Yg, best_L


class FullMechanismAnimator:
    def __init__(self, drop_c, u, v, Xg, Yg, L_coupler, radius, omega=10.0):
        self.frames = 150
        self.history = []
        self.theta_history = []

        self.drop_c = drop_c
        self.u, self.v = u, v
        self.Xg, self.Yg = Xg, Yg
        self.L_coupler = L_coupler
        self.radius = radius
        self.omega_anim = omega

        safe_shift = self.radius + self.drop_c + (self.radius * 0.5)
        self.L_lever_drive = self.drop_c
        self.L_lever_visual = self.L_lever_drive * 1.3

        self.crank_r = max(self.radius * 0.1, self.drop_c * 0.4)
        self.ground_d = self.crank_r * 2.0

        self.G_lever = (-safe_shift, self.drop_c)
        self.G_crank = (-safe_shift - self.ground_d, self.drop_c)
        self.Y_slider_offset = self.G_lever[1] + (self.L_lever_drive * 0.5)

    def analytical_kinematics(self, phi):
        Ax = self.G_crank[0] + self.crank_r * math.cos(phi)
        Ay = self.G_crank[1] + self.crank_r * math.sin(phi)

        beta = math.atan2(Ay - self.G_lever[1], Ax - self.G_lever[0])

        Bx_vis = self.G_lever[0] + self.L_lever_visual * math.cos(beta)
        By_vis = self.G_lever[1] + self.L_lever_visual * math.sin(beta)

        Bx_pin = self.G_lever[0] + self.L_lever_drive * math.cos(beta)
        By_pin = self.G_lever[1] + self.L_lever_drive * math.sin(beta)

        Y_slider = By_pin - self.Y_slider_offset

        C1x, C1y, R1 = 0.0, Y_slider, math.hypot(self.u, self.v)
        C2x, C2y, R2 = self.Xg, self.Yg, self.L_coupler

        dist = math.hypot(C1x - C2x, C1y - C2y)

        if dist <= (R1 + R2) and dist >= abs(R1 - R2):
            a = (R1**2 - R2**2 + dist**2) / (2 * dist)
            h_sq = max(0, R1**2 - a**2)
            h = math.sqrt(h_sq)

            P2x = C1x + a * (C2x - C1x) / dist
            P2y = C1y + a * (C2y - C1y) / dist

            Px_sol1 = P2x + h * (C2y - C1y) / dist
            Py_sol1 = P2y - h * (C2x - C1x) / dist

            Px_sol2 = P2x - h * (C2y - C1y) / dist
            Py_sol2 = P2y + h * (C2x - C1x) / dist

            local_pin_angle = math.atan2(self.v, self.u)
            theta1_obj = math.atan2(Py_sol1 - Y_slider, Px_sol1) - local_pin_angle
            theta2_obj = math.atan2(Py_sol2 - Y_slider, Px_sol2) - local_pin_angle

            theta1_obj = (theta1_obj + math.pi) % (2 * math.pi) - math.pi
            theta2_obj = (theta2_obj + math.pi) % (2 * math.pi) - math.pi

            if not self.theta_history:
                px1 = self.u * math.cos(theta1_obj) - self.v * math.sin(theta1_obj)
                px2 = self.u * math.cos(theta2_obj) - self.v * math.sin(theta2_obj)
                if px1 <= 0 and px2 > 0:
                    theta_exact = theta1_obj
                elif px2 <= 0 and px1 > 0:
                    theta_exact = theta2_obj
                else:
                    theta_exact = theta1_obj if abs(theta1_obj) < abs(theta2_obj) else theta2_obj
            else:
                prev = self.theta_history[-1]
                diff1 = abs((theta1_obj - prev + math.pi) % (2 * math.pi) - math.pi)
                diff2 = abs((theta2_obj - prev + math.pi) % (2 * math.pi) - math.pi)
                theta_exact = theta1_obj if diff1 < diff2 else theta2_obj
        else:
            if len(self.theta_history) >= 2:
                theta_exact = 2 * self.theta_history[-1] - self.theta_history[-2]
            else:
                theta_exact = self.theta_history[-1] if self.theta_history else 0.0

        Px = self.u * math.cos(theta_exact) - self.v * math.sin(theta_exact)
        Py = Y_slider + self.u * math.sin(theta_exact) + self.v * math.cos(theta_exact)

        self.theta_history.append(theta_exact)
        return {
            'joints_x': [Ax, Bx_vis, Bx_pin, Px],
            'joints_y': [Ay, By_vis, By_pin, Py],
            'slider_y': Y_slider,
            'theta': theta_exact,
        }

    def generate_history(self):
        self.history = []
        self.theta_history = []
        phi_start = math.pi / 3.0
        phis = np.linspace(phi_start, phi_start + 2 * math.pi, self.frames, endpoint=False)
        for phi in phis:
            self.history.append(self.analytical_kinematics(phi))

    def create_animation(self):
        self.generate_history()

        fig, ax = plt.subplots(figsize=(12, 8))
        ax.set_aspect('equal')

        pad_view = max(self.radius * 0.5, 20.0)
        extreme_x = [self.G_crank[0] - pad_view, max(self.Xg, self.radius) + pad_view]
        extreme_y = [
            min(-(self.drop_c + self.radius + pad_view), self.Yg - pad_view),
            max(self.G_lever[1] + self.L_lever_visual + pad_view, self.Yg + pad_view)
        ]
        ax.set_xlim(extreme_x[0], extreme_x[1])
        ax.set_ylim(extreme_y[0], extreme_y[1])

        ax.set_facecolor('#f8f9fa')
        ax.grid(True, linestyle='-', color='white', lw=1.5, zorder=0)

        # --- TIMER ADDITION: Initialize text object ---
        time_text = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=14,
                            family='monospace', verticalalignment='top',
                            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.9), zorder=20)

        # --- TIMER ADDITION: Pre-calculate time limits based on omega ---
        total_cycle_time = (2 * math.pi) / self.omega_anim
        time_per_frame = total_cycle_time / self.frames

        slider_y_vals = [k['slider_y'] for k in self.history]
        min_sy = min(slider_y_vals)
        max_sy = max(slider_y_vals)

        # Dynamically calculate the exact turning point to match the dynamic analysis script
        forward_end_frame = int(np.argmin(slider_y_vals))

        slot_width = self.radius * 0.20
        pad_val = slot_width * 0.1

        s_w = slot_width - (2 * pad_val)
        s_h = (slot_width * 2.0) - (2 * pad_val)

        margin = slot_width * 0.2
        t_y_start = min_sy - (s_h / 2) - margin
        t_y_end = max_sy + (s_h / 2) + margin
        t_height = t_y_end - t_y_start

        t_width_inner = slot_width
        t_width_outer = slot_width * 2.5

        track_bg = patches.Rectangle(
            (-t_width_outer / 2, t_y_start), t_width_outer, t_height,
            color='#cccccc', zorder=1
        )
        track_fg = patches.Rectangle(
            (-t_width_inner / 2, t_y_start + margin), t_width_inner, t_height - (2 * margin),
            color='#222222', zorder=2
        )

        ax.add_patch(track_bg)
        ax.add_patch(track_fg)

        ax.plot(self.G_crank[0], self.G_crank[1], 'k^', markersize=12, zorder=10)
        ax.plot(self.G_lever[0], self.G_lever[1], 'k^', markersize=12, zorder=10)
        ax.plot(self.Xg, self.Yg, 'k^', markersize=12, zorder=10)

        crank_line, = ax.plot(
            [], [], color='#555555', lw=12, solid_capstyle='round', zorder=3,
            path_effects=[pe.Stroke(linewidth=15, foreground='black'), pe.Normal()]
        )
        lever_line, = ax.plot(
            [], [], color='#888888', lw=12, solid_capstyle='round', zorder=4,
            path_effects=[pe.Stroke(linewidth=15, foreground='black'), pe.Normal()]
        )
        transfer_link, = ax.plot(
            [], [], color='#777777', lw=7, solid_capstyle='round', zorder=3,
            path_effects=[pe.Stroke(linewidth=10, foreground='black'), pe.Normal()]
        )
        coupler_line, = ax.plot(
            [], [], color='#AAAAAA', lw=10, solid_capstyle='round', zorder=3,
            path_effects=[pe.Stroke(linewidth=13, foreground='black'), pe.Normal()]
        )

        crank_slider, = ax.plot(
            [], [], marker='o', color='#e0e0e0', markeredgecolor='black',
            markeredgewidth=2, markersize=13, zorder=6
        )

        slider_rect = patches.FancyBboxPatch(
            (-s_w / 2, -s_h / 2), s_w, s_h,
            boxstyle=f"round,pad={pad_val}",
            facecolor='#ff9900', edgecolor='black', lw=2.0, zorder=6
        )
        ax.add_patch(slider_rect)

        circle_pin, = ax.plot(
            [], [], marker='o', color='#444444', markeredgecolor='white',
            markeredgewidth=1.5, markersize=10, zorder=12
        )

        pin_radius_line, = ax.plot([], [], color='#444444', lw=2.5, linestyle='--', zorder=11)
        joints_plot, = ax.plot([], [], 'o', color='white', markeredgecolor='black',
                               markeredgewidth=1.2, markersize=5, zorder=10)

        wedge_patch = patches.Wedge(
            (0, 0), self.radius, 90, 360,
            facecolor='#aec7e8', edgecolor='#1f77b4', lw=2, alpha=0.8, zorder=2
        )
        ax.add_patch(wedge_patch)

        def init():
            slider_rect.set_bounds(-s_w / 2, -s_h / 2, s_w, s_h)
            time_text.set_text('')
            return (
                crank_line, lever_line, transfer_link, coupler_line,
                crank_slider, joints_plot, wedge_patch, circle_pin,
                pin_radius_line, slider_rect, time_text
            )

        def animate(i):
            k = self.history[i]

            crank_line.set_data([self.G_crank[0], k['joints_x'][0]], [self.G_crank[1], k['joints_y'][0]])
            lever_line.set_data([self.G_lever[0], k['joints_x'][1]], [self.G_lever[1], k['joints_y'][1]])
            transfer_link.set_data([k['joints_x'][2], 0], [k['joints_y'][2], k['slider_y']])
            coupler_line.set_data([self.Xg, k['joints_x'][3]], [self.Yg, k['joints_y'][3]])

            crank_slider.set_data([k['joints_x'][0]], [k['joints_y'][0]])
            slider_rect.set_bounds(-s_w / 2, k['slider_y'] - s_h / 2, s_w, s_h)

            circle_pin.set_data([k['joints_x'][3]], [k['joints_y'][3]])
            pin_radius_line.set_data([0, k['joints_x'][3]], [k['slider_y'], k['joints_y'][3]])

            all_pins_x = [self.G_crank[0], self.G_lever[0], self.Xg, k['joints_x'][0], k['joints_x'][2], 0]
            all_pins_y = [self.G_crank[1], self.G_lever[1], self.Yg, k['joints_y'][0], k['joints_y'][2], k['slider_y']]
            joints_plot.set_data(all_pins_x, all_pins_y)

            wedge_angle_deg = math.degrees(k['theta'])
            wedge_patch.set_center((0, k['slider_y']))
            wedge_patch.set_theta1(90 + wedge_angle_deg)
            wedge_patch.set_theta2(360 + wedge_angle_deg)

            # --- TIMER ADDITION: 4-decimal precision strictly ensures visual separation ---
            current_time = i * (total_cycle_time / (self.frames - 1))
            if i <= forward_end_frame:
                time_text.set_text(f'Time: {current_time:06.4f} s \nForward Stroke...')
                time_text.set_color('black')
                time_text.set_fontweight('normal')
            else:
                time_text.set_text(f'Time: {current_time:06.4f} s \nReturn Stroke...')
                time_text.set_color('black')
                time_text.set_fontweight('normal')

            return (
                crank_line, lever_line, transfer_link, coupler_line,
                crank_slider, joints_plot, wedge_patch, circle_pin,
                pin_radius_line, slider_rect, time_text
            )

        interval = max(12, int(2000 * math.pi / (self.omega_anim * self.frames)))
        anim = animation.FuncAnimation(
            fig, animate, init_func=init, frames=self.frames, interval=interval, blit=True
        )
        return anim

In [ ]:
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 50.0

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D
import matplotlib.patheffects as PathEffects

COLORS = {
    'A_total': '#D62828',
    'A_comp': '#F9A03F',
    'BA_comp': '#48CAE4',
    'Coriolis': '#0077B6',
    'BA_total': '#03045E',
    'B_total': '#2A9D8F',
    'B_comp': '#8AB17D',
    'Slider': '#E76F51',
    'Coupler': '#9D4EDD',
    'Origin': '#000000'
}

class KinematicsEngine:
    def __init__(self, drop_c, u, v, Xg, Yg, L_coupler, radius):
        self.drop_c = drop_c
        self.u = u
        self.v = v
        self.Xg = Xg
        self.Yg = Yg
        self.L_coupler = L_coupler
        self.radius = radius

        safe_shift = radius + drop_c + radius * 0.5
        self.L_lever_drive = drop_c
        self.L_lever_visual = drop_c * 1.3
        self.crank_r = max(radius * 0.1, drop_c * 0.4)
        self.ground_d = self.crank_r * 2.0
        self.G_lever = (-safe_shift, drop_c)
        self.G_crank = (-safe_shift - self.ground_d, drop_c)
        self.Y_slider_offset = self.G_lever[1] + self.L_lever_drive * 0.5
        self.theta_history = []

    def reset(self):
        self.theta_history = []

    def step(self, phi):
        u, v = self.u, self.v
        Ax = self.G_crank[0] + self.crank_r * math.cos(phi)
        Ay = self.G_crank[1] + self.crank_r * math.sin(phi)

        beta = math.atan2(Ay - self.G_lever[1], Ax - self.G_lever[0])

        Bx_pin = self.G_lever[0] + self.L_lever_drive * math.cos(beta)
        By_pin = self.G_lever[1] + self.L_lever_drive * math.sin(beta)
        Y_slider = By_pin - self.Y_slider_offset

        C1x, C1y, R1 = 0.0, Y_slider, math.hypot(u, v)
        C2x, C2y, R2 = self.Xg, self.Yg, self.L_coupler
        dist = math.hypot(C1x - C2x, C1y - C2y)

        if dist <= (R1 + R2) and dist >= abs(R1 - R2):
            a = (R1**2 - R2**2 + dist**2) / (2 * dist)
            h = math.sqrt(max(0, R1**2 - a**2))
            P2x = C1x + a * (C2x - C1x) / dist
            P2y = C1y + a * (C2y - C1y) / dist

            Px1 = P2x + h * (C2y - C1y) / dist
            Py1 = P2y - h * (C2x - C1x) / dist

            Px2 = P2x - h * (C2y - C1y) / dist
            Py2 = P2y + h * (C2x - C1x) / dist

            lpa = math.atan2(v, u)
            th1 = ((math.atan2(Py1 - Y_slider, Px1) - lpa) + math.pi) % (2 * math.pi) - math.pi
            th2 = ((math.atan2(Py2 - Y_slider, Px2) - lpa) + math.pi) % (2 * math.pi) - math.pi

            if not self.theta_history:
                px1 = u * math.cos(th1) - v * math.sin(th1)
                px2 = u * math.cos(th2) - v * math.sin(th2)
                if px1 <= 0 and px2 > 0:
                    theta_exact = th1
                elif px2 <= 0 and px1 > 0:
                    theta_exact = th2
                else:
                    theta_exact = th1 if abs(th1) < abs(th2) else th2
            else:
                prev = self.theta_history[-1]
                d1 = abs((th1 - prev + math.pi) % (2 * math.pi) - math.pi)
                d2 = abs((th2 - prev + math.pi) % (2 * math.pi) - math.pi)
                theta_exact = th1 if d1 < d2 else th2
        else:
            theta_exact = (
                2 * self.theta_history[-1] - self.theta_history[-2]
                if len(self.theta_history) >= 2 else 0.0
            )

        Px = u * math.cos(theta_exact) - v * math.sin(theta_exact)
        Py = Y_slider + u * math.sin(theta_exact) + v * math.cos(theta_exact)
        self.theta_history.append(theta_exact)

        return dict(
            Ax=Ax, Ay=Ay, Bx=Bx_pin, By=By_pin,
            slider_x=0.0, slider_y=Y_slider,
            Px=Px, Py=Py,
            theta=theta_exact, beta=beta, phi=phi
        )


def compute_kinematics_history(engine, omega2, frames=150):
    engine.reset()
    phi_start = math.pi / 3.0
    dt = (2 * math.pi / omega2) / frames
    phis = np.linspace(phi_start, phi_start + 2 * math.pi, frames, endpoint=False)

    phis_ext = np.concatenate([[phis[-2], phis[-1]], phis, [phis[0], phis[1]]])
    engine.reset()
    states_ext = [engine.step(phi) for phi in phis_ext]

    keys = ['Ax', 'Ay', 'Bx', 'By', 'slider_x', 'slider_y', 'Px', 'Py']
    history = []

    for i in range(2, frames + 2):
        sp, sc, sn = states_ext[i - 1], states_ext[i], states_ext[i + 1]
        entry = dict(sc)
        for k in keys:
            entry['V_' + k] = (sn[k] - sp[k]) / (2 * dt)
            entry['A_' + k] = (sn[k] - 2 * sc[k] + sp[k]) / dt**2
        history.append(entry)

    return history, dt


def find_best_start_frame(history):
    best_idx, best_spread = 0, -1.0
    for i, h in enumerate(history):
        pts_x = [0, h['V_Ax'], h['V_Bx'], h['V_Px']]
        pts_y = [0, h['V_Ay'], h['V_By'], h['V_slider_y'], h['V_Py']]
        spread = max(max(pts_x) - min(pts_x), max(pts_y) - min(pts_y))
        if spread > best_spread:
            best_spread, best_idx = spread, i
    return best_idx


def draw_aesthetic_vector(ax, start, end, color, label='', linestyle='-', zorder=3, is_total=False, custom_lw=None):
    if abs(start[0] - end[0]) < 1e-6 and abs(start[1] - end[1]) < 1e-6:
        ax.plot(start[0], start[1], 'o', color=color, markersize=4, zorder=zorder)
        if label:
            txt = ax.text(
                start[0], start[1], label, color=color, fontsize=10,
                ha='center', va='bottom', fontweight='bold', zorder=10
            )
            txt.set_path_effects([PathEffects.withStroke(linewidth=3, foreground='w')])
        return

    lw = custom_lw if custom_lw is not None else (2.5 if is_total else 1.5)
    arrow_scale = 15 if is_total else 10

    arrow = FancyArrowPatch(
        posA=start, posB=end,
        arrowstyle=f'-|>,head_length={arrow_scale},head_width={arrow_scale * 0.6}',
        color=color, linewidth=lw, linestyle=linestyle, zorder=zorder
    )
    ax.add_patch(arrow)

    if label:
        mid_x = (start[0] + end[0]) / 2
        mid_y = (start[1] + end[1]) / 2
        txt = ax.text(
            mid_x, mid_y, label, color=color, fontsize=10,
            ha='center', va='bottom', fontweight='bold', zorder=10
        )
        txt.set_path_effects([PathEffects.withStroke(linewidth=3, foreground='w')])


def _set_tight_limits(ax, xs, ys, pad_frac=0.35):
    xlo, xhi = min(xs), max(xs)
    ylo, yhi = min(ys), max(ys)
    span = max(xhi - xlo, yhi - ylo, 0.05)
    cx, cy = (xhi + xlo) / 2, (yhi + ylo) / 2
    half = span / 2 * (1 + pad_frac)
    ax.set_xlim(cx - half, cx + half)
    ax.set_ylim(cy - half, cy + half)
    ax.set_aspect('equal', adjustable='box')


def _dot_and_tag(ax, px_, py_, lbl, col, zorder=8):
    ax.plot(px_, py_, 'o', color=col, markersize=5, zorder=zorder)
    txt = ax.annotate(
        lbl, xy=(px_, py_), xytext=(5, 5), textcoords='offset points',
        fontsize=8.5, color=col, fontweight='bold', zorder=zorder + 1
    )
    txt.set_path_effects([PathEffects.withStroke(linewidth=2, foreground='w')])


_VEL_GLOSSARY = (
    "o  = Vel. Pole (Origin)\n"
    "a  = Tip of $V_A$\n"
    "b  = Tip of $V_B$\n"
    "s  = Tip of $V_{slider}$\n"
    "p  = Tip of $V_P$"
)

_ACC_GLOSSARY = (
    "o' = Accel Pole (Origin)\n"
    "a' = Tip of $A_A$\n"
    "b' = Tip of $A_B$\n"
    "s' = Tip of $A_{slider}$\n"
    "p' = Tip of $A_P$\n"
    "$n_a$: Tip of centripetal $A^n_A$\n"
    "$n_b$: Tip of centripetal $A^n_{B/A}$"
)


def _add_glossary(ax, text, title="Nodes"):
    full_text = f"── {title} ──\n" + text
    ax.text(
        0.98, 0.02, full_text, transform=ax.transAxes,
        fontsize=7.5, va='bottom', ha='right',
        family='sans-serif', linespacing=1.4,
        bbox=dict(boxstyle='round,pad=0.6', fc='#ffffff', ec='#bdc3c7', lw=1.2, alpha=0.95)
    )


def draw_velocity_polygon(ax, frame_data, scale=1.0):
    ax.cla()
    ax.set_facecolor('#f8f9fa')
    ax.grid(True, color='white', lw=1.2, zorder=0)

    VAx = frame_data['V_Ax'];  VAy = frame_data['V_Ay']
    VBx = frame_data['V_Bx'];  VBy = frame_data['V_By']
    Vs  = frame_data['V_slider_y']
    VPx = frame_data['V_Px'];  VPy = frame_data['V_Py']

    ox = oy = 0.0
    ax_t = VAx * scale;   ay_t = VAy * scale
    bx_t = VBx * scale;   by_t = VBy * scale
    sx_t = 0.0;           sy_t = Vs * scale
    px_t = VPx * scale;   py_t = VPy * scale

    all_x = [ox, ax_t, bx_t, sx_t, px_t]
    all_y = [oy, ay_t, by_t, sy_t, py_t]

    draw_aesthetic_vector(ax, (ox, oy), (ax_t, ay_t), COLORS['A_total'], '$V_A$', zorder=5, is_total=True)
    draw_aesthetic_vector(ax, (ax_t, ay_t), (bx_t, by_t), COLORS['BA_total'], '$V_{B/A}$', zorder=5, is_total=True)
    draw_aesthetic_vector(ax, (ox, oy), (bx_t, by_t), COLORS['B_total'], '$V_B$', zorder=4, is_total=True)
    draw_aesthetic_vector(ax, (ox, oy), (sx_t, sy_t), COLORS['Slider'], '$V_{slider}$', zorder=4, is_total=True)
    draw_aesthetic_vector(ax, (ox, oy), (px_t, py_t), COLORS['Coupler'], '$V_P$', zorder=4, is_total=True)

    for px_, py_, lbl, col in [
        (ox, oy, 'o', COLORS['Origin']),
        (ax_t, ay_t, 'a', COLORS['A_total']),
        (bx_t, by_t, 'b', COLORS['B_total']),
        (sx_t, sy_t, 's', COLORS['Slider']),
        (px_t, py_t, 'p', COLORS['Coupler']),
    ]:
        _dot_and_tag(ax, px_, py_, lbl, col)

    _set_tight_limits(ax, all_x, all_y, pad_frac=0.35)
    ax.set_title('Velocity Polygon', fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('$V_x$  (m/s)')
    ax.set_ylabel('$V_y$  (m/s)')

    ax.text(
        0.02, 0.98,
        f"$|V_A|$: {math.hypot(VAx, VAy):.2f}\n"
        f"$|V_B|$: {math.hypot(VBx, VBy):.2f}\n"
        f"$|V_P|$: {math.hypot(VPx, VPy):.2f}\n"
        f"$V_{{slider}}$: {Vs:.2f}",
        transform=ax.transAxes, fontsize=8, va='top', ha='left', linespacing=1.4,
        bbox=dict(boxstyle='round,pad=0.5', fc='#ffffff', ec='#bdc3c7', alpha=0.95)
    )

    _add_glossary(ax, _VEL_GLOSSARY, "Velocity Nodes")

    legend_elements = [
        Line2D([0], [0], color=COLORS['A_total'], lw=2.5, label='Crank (A)'),
        Line2D([0], [0], color=COLORS['BA_total'], lw=2.5, label='Rel. (B/A)'),
        Line2D([0], [0], color=COLORS['B_total'], lw=2.5, label='Lever (B)'),
        Line2D([0], [0], color=COLORS['Slider'], lw=2.5, label='Slider'),
        Line2D([0], [0], color=COLORS['Coupler'], lw=2.5, label='Coupler (P)'),
    ]
    leg = ax.legend(handles=legend_elements, loc='lower left', fontsize=7.5,
                    framealpha=0.95, edgecolor='#bdc3c7')
    leg.set_title("Vector Colors", prop={'size': 8, 'weight': 'bold'})


def draw_acceleration_polygon(ax, h, scale=1.0):
    ax.cla()
    ax.set_facecolor('#f8f9fa')
    ax.grid(True, color='white', lw=1.2, zorder=0)
    ax.set_title('Acceleration Polygon (Coriolis included)', fontsize=12, fontweight='bold', pad=10)

    phi = h['phi']
    beta = h['beta']
    om2 = h['omega2']
    r2 = h['r2']
    L_lev = h['L_lever']

    aA = np.array([h['A_Ax'], h['A_Ay']])
    aB = np.array([h['A_Bx'], h['A_By']])
    aS = np.array([0.0, h['A_slider_y']])
    aP = np.array([h['A_Px'], h['A_Py']])
    VA = np.array([h['V_Ax'], h['V_Ay']])
    VB = np.array([h['V_Bx'], h['V_By']])

    e3 = np.array([math.cos(beta), math.sin(beta)])
    e3p = np.array([-math.sin(beta), math.cos(beta)])
    er2 = np.array([math.cos(phi), math.sin(phi)])

    om3 = np.dot(VB, e3p) / max(L_lev, 1e-6)
    s_dot = np.dot(VA, e3)

    aA_n = -om2**2 * r2 * er2
    aA_t = aA - aA_n

    Aba = aB - aA
    a_coriolis = 2.0 * om3 * s_dot * e3p
    Aba_t = np.dot(Aba, e3) * e3
    Aba_n = Aba - Aba_t - a_coriolis

    O = np.zeros(2)
    na = O + aA_n * scale
    ap = na + aA_t * scale

    p1 = ap + Aba_n * scale
    p2 = p1 + Aba_t * scale
    bp = p2 + a_coriolis * scale

    sp = O + aS * scale
    pp = O + aP * scale

    draw_aesthetic_vector(ax, O, na, COLORS['A_comp'], '$a_A^n$', linestyle='--', zorder=2, custom_lw=3.5)
    draw_aesthetic_vector(ax, na, ap, COLORS['A_comp'], '$a_A^t$', linestyle=':', zorder=3, custom_lw=2.5)
    draw_aesthetic_vector(ax, O, ap, COLORS['A_total'], '$a_A$', zorder=6, is_total=True, custom_lw=1.5)
    _dot_and_tag(ax, na[0], na[1], "n_a", COLORS['A_comp'], zorder=7)
    _dot_and_tag(ax, ap[0], ap[1], "a'", COLORS['A_total'], zorder=7)

    draw_aesthetic_vector(ax, ap, p1, COLORS['BA_comp'], '$a_{B/A}^n$', linestyle='--', zorder=3)
    draw_aesthetic_vector(ax, p1, p2, COLORS['BA_comp'], '$a_{B/A}^t$', linestyle='--', zorder=3)
    draw_aesthetic_vector(ax, p2, bp, COLORS['Coriolis'], '$a_{cor}$', linestyle='--', zorder=3)
    draw_aesthetic_vector(ax, ap, bp, COLORS['BA_total'], '$a_{B/A}$', zorder=5, is_total=True)
    _dot_and_tag(ax, bp[0], bp[1], "b'", COLORS['BA_total'], zorder=7)

    draw_aesthetic_vector(ax, O, bp, COLORS['B_total'], '$a_B$', zorder=4, is_total=True)
    draw_aesthetic_vector(ax, O, sp, COLORS['Slider'], '$a_{slider}$', zorder=4, is_total=True)
    draw_aesthetic_vector(ax, O, pp, COLORS['Coupler'], '$a_P$', zorder=4, is_total=True)

    _dot_and_tag(ax, sp[0], sp[1], "s'", COLORS['Slider'], zorder=7)
    _dot_and_tag(ax, pp[0], pp[1], "p'", COLORS['Coupler'], zorder=7)
    _dot_and_tag(ax, O[0], O[1], "o'", COLORS['Origin'], zorder=7)

    all_pts = [O, na, ap, p1, p2, bp, sp, pp]
    _set_tight_limits(ax, [p[0] for p in all_pts], [p[1] for p in all_pts], pad_frac=0.35)

    ax.set_xlabel('$a_x$ (mm/s²)')
    ax.set_ylabel('$a_y$ (mm/s²)')

    ax.text(
        0.02, 0.98,
        f"$|a_A|$: {np.linalg.norm(aA):.2f}\n"
        f"$|a_B|$: {np.linalg.norm(aB):.2f}\n"
        f"$|a_{{B/A}}|$: {np.linalg.norm(Aba):.2f}\n"
        f"$|a_P|$: {np.linalg.norm(aP):.2f}\n"
        f"$a_{{slider}}$: {aS[1]:.2f}\n"
        f"Coriolis mag: {np.linalg.norm(a_coriolis):.2f}",
        transform=ax.transAxes, fontsize=8, va='top', ha='left', linespacing=1.4,
        bbox=dict(boxstyle='round,pad=0.5', fc='#ffffff', ec='#bdc3c7', alpha=0.95)
    )

    legend_elements = [
        Line2D([0], [0], color=COLORS['A_total'], lw=2.5, label='Crank (A)'),
        Line2D([0], [0], color=COLORS['A_comp'], lw=1.5, linestyle='--', label='Crank Comp.'),
        Line2D([0], [0], color=COLORS['BA_total'], lw=2.5, label='Rel. (B/A)'),
        Line2D([0], [0], color=COLORS['BA_comp'], lw=1.5, linestyle='--', label='Rel. Comp.'),
        Line2D([0], [0], color=COLORS['Coriolis'], lw=1.5, linestyle='--', label='Coriolis'),
        Line2D([0], [0], color=COLORS['B_total'], lw=2.5, label='Lever (B)'),
        Line2D([0], [0], color=COLORS['Slider'], lw=2.5, label='Slider'),
        Line2D([0], [0], color=COLORS['Coupler'], lw=2.5, label='Coupler (P)'),
    ]
    leg = ax.legend(handles=legend_elements, loc='lower left', fontsize=7.5,
                    framealpha=0.95, edgecolor='#bdc3c7')
    leg.set_title("Vector Colors", prop={'size': 8, 'weight': 'bold'})

    _add_glossary(ax, _ACC_GLOSSARY, "Accel Nodes")

In [ ]:
def run_combined_animation(radius, drop_c, pin_u, pin_v,
                           Xg, Yg, L_coupler, omega2=10.0):

    engine  = KinematicsEngine(drop_c, pin_u, pin_v, Xg, Yg, L_coupler, radius)
    frames  = 150
    history, dt = compute_kinematics_history(engine, omega2, frames)

    # ── Inject every key that draw_acceleration_polygon needs ─────────────────
    for h in history:
        h.setdefault('omega2',  omega2)
        h.setdefault('alpha2',  0.0)               # constant-speed crank → α₂ = 0
        h.setdefault('r2',      engine.crank_r)    # crank radius stored in engine
        h.setdefault('L_lever', engine.L_lever_drive)  # lever length stored in engine

    # ── Rotate history so most "spread" frame comes first ─────────────────────
    start   = find_best_start_frame(history)
    history = history[start:] + history[:start]

    # ── Auto-scale polygons to ~80% of subplot ────────────────────────────────
    max_v = max(math.hypot(h['V_Ax'], h['V_Ay']) for h in history)
    max_a = max(math.hypot(h['A_Ax'], h['A_Ay']) for h in history)
    v_scale = 0.8 / (max_v + 1e-9)
    a_scale = 0.8 / (max_a + 1e-9)

    fig, (ax_vel, ax_acc) = plt.subplots(1, 2, figsize=(14, 6))
    fig.patch.set_facecolor('#f0f0f0')

    def init():
        draw_velocity_polygon(ax_vel,  history[0], scale=v_scale)
        draw_acceleration_polygon(ax_acc, history[0], scale=a_scale)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        return []

    def animate(i):
        draw_velocity_polygon(ax_vel,  history[i], scale=v_scale)
        draw_acceleration_polygon(ax_acc, history[i], scale=a_scale)
        fig.suptitle(
            f'Mechanism Kinematics  ·  φ = {math.degrees(history[i]["phi"]):.1f}°'
            f'  ·  frame {i+1}/{frames}',
            fontsize=12, fontweight='bold')
        return []

    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=frames, interval=30, blit=False)
    plt.close(fig)
    return anim

In [ ]:


def generate_static_plots(history, density, omega_crank, radius, L_coupler):
    """
    Generates static plots for total inertial force and input torque
    across the full 360-degree crank rotation.
    """
    scale = 0.001
    A = 0.0001  # uniform cross-section area assumption in m^2

    m_coupler = A * (L_coupler * scale) * density
    vol_slider = 0.0001
    m_slider = vol_slider * density

    r_obj = radius * scale
    m_obj = density * 0.01 * (math.pi * r_obj*r_obj)
    I_obj = m_obj * (r_obj**2) * ((3/8) - (8 /(27 * (math.pi**2))))

    angles = []
    torques = []
    forces = []

    for h in history:
        phi = h['phi']
        angles.append(math.degrees(phi) % 360)

        Asy = h['A_slider_y'] * scale
        Apx = h['A_Px'] * scale
        Apy = h['A_Py'] * scale

        Vsy = h['V_slider_y'] * scale
        Vpx = h['V_Px'] * scale
        Vpy = h['V_Py'] * scale

        F_slider_y = m_slider * Asy
        F_obj_x = m_obj * Apx
        F_obj_y = m_obj * Apy

        total_force = math.hypot(F_obj_x, F_obj_y + F_slider_y)
        forces.append(total_force)

        # Virtual power approximation for input torque
        power = (F_slider_y * Vsy) + (F_obj_x * Vpx) + (F_obj_y * Vpy)
        t_req = power / omega_crank
        torques.append(abs(t_req))

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
    fig.patch.set_facecolor('#f8f9fa')

    sorted_data = sorted(zip(angles, forces, torques))
    ang_sorted, f_sorted, t_sorted = zip(*sorted_data)

    ax1.plot(ang_sorted, f_sorted, lw=2.5)
    ax1.set_title("Total Inertial Force vs Input Crank Angle", fontsize=12, fontweight='bold')
    ax1.set_xlabel("Crank Angle φ (degrees)")
    ax1.set_ylabel("Force (N)")
    ax1.grid(True, linestyle='--', alpha=0.7)
    ax1.set_xlim(0, 360)

    ax2.plot(ang_sorted, t_sorted, lw=2.5)

    max_t = max(t_sorted)
    max_idx = t_sorted.index(max_t)
    max_angle = ang_sorted[max_idx]

    ax2.plot(max_angle, max_t, 'ko', markersize=8)
    ax2.annotate(
        f'Highest Torque: {max_t:.4f} N·m\n@ {max_angle:.1f}°',
        xy=(max_angle, max_t),
        xytext=(max_angle + 15, max_t * 0.8),
        bbox=dict(boxstyle='round,pad=0.3', fc='#ffeaa7', ec='#b2bec3', lw=1.5),
        arrowprops=dict(facecolor='black', shrink=0.05, width=2, headwidth=8)
    )

    ax2.set_title(
        f"Required Input Torque vs Crank Angle (Material Density: {density} kg/m³)",
        fontsize=12, fontweight='bold'
    )
    ax2.set_xlabel("Crank Angle φ (degrees)")
    ax2.set_ylabel("Torque (N·m)")
    ax2.grid(True, linestyle='--', alpha=0.7)
    ax2.set_xlim(0, 360)

    plt.tight_layout()
    plt.show()

In [ ]:
#Block-8
import math
import numpy as np
import matplotlib.pyplot as plt

def run_stress_test_and_dynamics(history, engine, density, E_mod, Yield_S, omega_crank, radius, L_coupler):
    """
    Advanced Dynamic Analysis & Stress Test satisfying Point 6 of the grading rubric.
    Calculates input torque (Virtual Power) and joint forces (Newton-Euler) to determine
    if the 10x10 mm members exceed the material's Yield Strength.
    """
    print("\n" + "═"*64)
    print(" BLOCK 8: ADVANCED DYNAMIC ANALYSIS & STRESS TEST ")
    print("═"*64)

    sc = 1e-3  # mm to m conversion
    A_cs_m2 = 100 * 1e-6  # 10x10 mm cross section in m^2
    A_cs_mm2 = 100.0      # 10x10 mm cross section in mm^2
    G_CONST = 9.81

    # Lengths in meters
    L_cr  = 2.0 * engine.crank_r * sc
    L_lev = engine.L_lever_drive * sc
    L_cpl = L_coupler * sc
    r_obj = radius * sc

    # Transfer link average length
    L_tr = float(np.mean([abs((h['By'] - h['slider_y']) * sc) for h in history]))

    # Masses (kg)
    m_cr  = density * A_cs_m2 * L_cr
    m_lev = density * A_cs_m2 * L_lev
    m_tr  = density * A_cs_m2 * L_tr
    m_cpl = density * A_cs_m2 * L_cpl
    m_sl  = density * 1e-6  # 1 cm³ point-mass slider
    m_obj = density * 0.01 * math.pi * r_obj *r_obj

    # Moment of Inertia for Lever (slender rod)
    I_lev_Gl = (1.0 / 3.0) * m_lev * (L_lev**2)

    angles = []
    T_drv = []
    JF_mtr = []
    JF_lev = []

    Gc_m = np.array([engine.G_crank[0] * sc, engine.G_crank[1] * sc])
    Gl_m = np.array([engine.G_lever[0] * sc, engine.G_lever[1] * sc])

    for h in history:
        angles.append(math.degrees(h['phi']) % 360)

        # Positions (m)
        Ax, Ay = h['Ax']*sc, h['Ay']*sc
        Bx, By = h['Bx']*sc, h['By']*sc
        sy = h['slider_y']*sc
        Px, Py = h['Px']*sc, h['Py']*sc
        beta = h['beta']

        # Accelerations (m/s²)
        aAx, aAy = h['A_Ax']*sc, h['A_Ay']*sc
        aBx, aBy = h['A_Bx']*sc, h['A_By']*sc
        asy = h['A_slider_y']*sc
        aPx, aPy = h['A_Px']*sc, h['A_Py']*sc

        # Velocities (m/s)
        vAx, vAy = h['V_Ax']*sc, h['V_Ay']*sc
        vBx, vBy = h['V_Bx']*sc, h['V_By']*sc
        vsy = h['V_slider_y']*sc
        vPx, vPy = h['V_Px']*sc, h['V_Py']*sc

        # CG accelerations
        a_CG_cr  = np.array([aAx/2, aAy/2])
        a_CG_lev = np.array([aBx/2, aBy/2])
        a_CG_tr  = np.array([aBx/2, (aBy + asy)/2])
        a_CG_cpl = np.array([aPx/2, aPy/2])

        # CG velocities
        v_CG_cr  = np.array([vAx/2, vAy/2])
        v_CG_lev = np.array([vBx/2, vBy/2])
        v_CG_tr  = np.array([vBx/2, (vBy+vsy)/2])
        v_CG_cpl = np.array([vPx/2, vPy/2])

        # Inertial forces (F = m*a)
        F_cr  = m_cr  * a_CG_cr
        F_lev = m_lev * a_CG_lev
        F_tr  = m_tr  * a_CG_tr
        F_sl  = np.array([0.0,  m_sl * asy])
        F_obj = np.array([m_obj * aPx, m_obj * aPy])
        F_cpl = m_cpl * a_CG_cpl

        # 1. Driving Torque via Virtual Power
        pwr = (np.dot(F_cr, v_CG_cr) + np.dot(F_lev, v_CG_lev) +
               np.dot(F_tr, v_CG_tr) + F_sl[1]*vsy +
               np.dot(F_obj, np.array([vPx, vPy])) + np.dot(F_cpl, v_CG_cpl))
        T_drv.append(pwr / omega_crank)

        # 2. Joint Forces via Newton-Euler
        e_perp = np.array([-math.sin(beta), math.cos(beta)])

        # Transfer-link reaction on lever at B_pin (opposes slider+obj inertia AND gravity)
        F_Bpin_y = -(m_sl + m_obj + m_tr) * (asy + G_CONST)
        F_Bpin = np.array([0.0, F_Bpin_y])
        F_Bpin_perp = np.dot(F_Bpin, e_perp)

        # Normal force from crank pin on lever
        L_lev_i = max(math.hypot(Bx - Gl_m[0], By - Gl_m[1]), 1e-9)
        alpha_lev_i = (aBx*e_perp[0] + aBy*e_perp[1]) / L_lev_i
        r_A_from_Gl = math.hypot(Ax - Gl_m[0], Ay - Gl_m[1])

        N_A = ((I_lev_Gl * alpha_lev_i - F_Bpin_perp * L_lev_i) / max(r_A_from_Gl, 1e-9))
        F_A1 = N_A * e_perp  # Force crank -> lever

        F_Gl = m_lev * a_CG_lev - F_A1 - F_Bpin  # Lever pivot reaction
        JF_lev.append(np.linalg.norm(F_Gl))

        F_Gc = m_cr * a_CG_cr + F_A1  # Crank motor pivot reaction
        JF_mtr.append(np.linalg.norm(F_Gc))

    # --- Plot Required Input Torque vs Angle ---
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('#f8f9fa')

    sort_idx = np.argsort(angles)
    ang_sorted = np.array(angles)[sort_idx]
    T_sorted = np.array(T_drv)[sort_idx]

    max_t = np.max(np.abs(T_sorted))
    max_idx = np.argmax(np.abs(T_sorted))
    max_angle = ang_sorted[max_idx]

    ax.plot(ang_sorted, T_sorted, lw=2.5, color='black')
    ax.plot(max_angle, T_sorted[max_idx], 'ro', markersize=8)
    ax.axhline(0, color='gray', lw=1, ls='--')

    ax.annotate(
        f'Highest Torque: {T_sorted[max_idx]:.4f} N·m\n@ {max_angle:.1f}°',
        xy=(max_angle, T_sorted[max_idx]),
        xytext=(max_angle + 15, T_sorted[max_idx] * 0.8),
        bbox=dict(boxstyle='round,pad=0.3', fc='#ffeaa7', ec='#b2bec3', lw=1.5),
        arrowprops=dict(facecolor='black', shrink=0.05, width=2, headwidth=8)
    )

    ax.set_title("Required Input Torque vs Crank Angle (Newton-Euler / Virtual Power)", fontsize=12, fontweight='bold')
    ax.set_xlabel("Crank Angle φ (degrees)")
    ax.set_ylabel("Driving Torque (N·m)")
    ax.grid(True, linestyle='--', alpha=0.7)
    ax.set_xlim(0, 360)
    plt.tight_layout()
    plt.show()

    # --- STRESS TEST MODULE ---
    print("\n─── STRESS TEST MODULE ──────────────────────────────────")
    print(f" Material Properties:")
    print(f"   Elastic Modulus : {E_mod} MPa")
    print(f"   Yield Strength  : {Yield_S} MPa")
    print(f"   Cross-Section   : 10x10 mm (Area = {A_cs_mm2} mm²)")

    # Max forces
    max_F_crank = np.max(JF_mtr)
    max_F_lever = np.max(JF_lev)

    # Approximate Axial Stress (MPa = N / mm^2)
    sigma_crank = max_F_crank / A_cs_mm2
    sigma_lever = max_F_lever / A_cs_mm2

    print("\n Calculated Maximum Joint Forces:")
    print(f"   Max Crank Force : {max_F_crank:.4f} N")
    print(f"   Max Lever Force : {max_F_lever:.4f} N")

    print("\n Calculated Approximate Axial Stresses:")
    print(f"   Crank Link: {sigma_crank:.4f} MPa")
    print(f"   Lever Link: {sigma_lever:.4f} MPa")

    failed = False
    print("\n --- FAILURE ANALYSIS ---")
    if sigma_crank > Yield_S:
        print("\033[1mWARNING: MECHANISM FAILED! Link [Crank] exceeded Yield Strength.\033[0m")
        failed = True
    if sigma_lever > Yield_S:
        print("\033[1mWARNING: MECHANISM FAILED! Link [Lever] exceeded Yield Strength.\033[0m")
        failed = True

    if not failed:
        print("\033[92m\033[1mSUCCESS: All members passed the stress test. Mechanism is safe.\033[0m")

    print("─────────────────────────────────────────────────────────\n")

In [ ]:
#Block-10
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

def run_extended_dynamic_analysis(radius, drop_b, drop_c, density, omega, pin_u, pin_v, Xg, Yg, L_coupler):
    """
    Modular implementation of the Extended Dynamic Analysis.
    Combines baseline geometric snapshot with the advanced Newton-Euler evaluation.
    """
    print("\n" + "═"*75)
    print(" BLOCK 10: EXTENDED DYNAMIC ANALYSIS (PLOTS & PROOFS) ")
    print("═"*75)

    sc = 1e-3  # mm → m conversion
    A_cs = 100.0 * 1e-6  # 10x10 mm cross section in m²

    # 1. Regenerate precise history using the current omega
    eng = KinematicsEngine(drop_c, pin_u, pin_v, Xg, Yg, L_coupler, radius)
    history, dt = compute_kinematics_history(eng, omega, frames=150)

    # Lengths in m
    L_cr  = 2.0 * eng.crank_r * sc
    L_lev = eng.L_lever_drive * sc
    L_cpl = L_coupler * sc
    r_obj = radius * sc

    # Transfer link: average vertical distance
    tr_len_arr = [abs((h['By'] - h['slider_y']) * sc) for h in history]
    L_tr = float(np.mean(tr_len_arr))

    # Masses & Inertias
    m_cr  = density * A_cs * L_cr
    m_lev = density * A_cs * L_lev
    m_tr  = density * A_cs * L_tr
    m_cpl = density * A_cs * L_cpl
    m_sl  = density * 1e-6
    m_obj = density * 0.01 * math.pi * r_obj *r_obj
    I_obj = m_obj * (r_obj**2) * ((3/8) - (8 /(27 * (math.pi**2))))

    print("── Material & Mass Summary ─────────────────────────────────────")
    print(f"  Material          : Aluminium,  ρ = {density:.0f} kg/m³")
    print(f"  Bar cross-section : {A_cs*1e6:.0f} mm²  (10 mm × 10 mm)")
    print(f"  m_crank  ({L_cr*1e3:.1f} mm)    : {m_cr*1e3:.3f} g")
    print(f"  m_lever  ({L_lev*1e3:.1f} mm)    : {m_lev*1e3:.3f} g")
    print(f"  m_transfer ({L_tr*1e3:.1f} mm avg): {m_tr*1e3:.3f} g")
    print(f"  m_coupler ({L_cpl*1e3:.1f} mm)   : {m_cpl*1e3:.3f} g")
    print(f"  m_slider (pt-mass): {m_sl*1e3:.3f} g")
    print(f"  m_object (arc r={r_obj*1e3:.0f} mm): {m_obj*1e3:.3f} g")
    print(f"  I_object (about CG): {I_obj*1e6:.4f} kg·m²")
    print("────────────────────────────────────────────────────────────────\n")

    # ── QUICK-RETURN PROOF ──
    sy_d   = np.array([h['slider_y'] for h in history])
    phi_d  = np.array([h['phi'] for h in history])
    iA_d   = int(np.argmax(sy_d))
    iC_d   = int(np.argmin(sy_d))

    phi_A_rad  = phi_d[iA_d] % (2 * math.pi)
    phi_C_rad  = phi_d[iC_d] % (2 * math.pi)
    delta_fwd  = (phi_C_rad - phi_A_rad) % (2 * math.pi)
    delta_ret  = 2 * math.pi - delta_fwd

    T_fwd      = delta_fwd / omega
    T_ret      = delta_ret / omega
    TR_actual  = T_fwd / (T_ret + 1e-12)

    print("── Quick-Return Stroke Proof ────────────────────────────────────")
    print(f"  Crank speed ω       : {omega:.2f} rad/s")
    print(f"  Full cycle period   : {2*math.pi/omega:.4f} s")
    print(f"  φ at State A        : {math.degrees(phi_A_rad):.1f}°")
    print(f"  φ at State C        : {math.degrees(phi_C_rad):.1f}°")
    print(f"  ┌ Forward stroke arc: {math.degrees(delta_fwd):.1f}°")
    print(f"  │ T_forward          : {T_fwd:.4f} s")
    print(f"  ├ Return stroke arc : {math.degrees(delta_ret):.1f}°")
    print(f"  │ T_return           : {T_ret:.4f} s")
    print(f"  └ Time Ratio (actual): {TR_actual:.3f}   ← target 2.000")
    if abs(TR_actual - 2.0) < 0.05:
        print("  ✓ Mechanism confirmed as 2:1 Quick-Return.")
    else:
        print(f"  ⚠ Actual TR deviates from 2.0")
    print("────────────────────────────────────────────────────────────────\n")

    # ── PER-FRAME DYNAMICS ──
    Gc_m = np.array([eng.G_crank[0] * sc, eng.G_crank[1] * sc])
    Gl_m = np.array([eng.G_lever[0] * sc, eng.G_lever[1] * sc])
    Xg_m = Xg * sc;  Yg_m = Yg * sc

    NN = len(history)
    phi_deg_all = np.array([h['phi'] for h in history]) * 180/math.pi % 360
    sort_idx    = np.argsort(phi_deg_all)
    phi_sorted  = phi_deg_all[sort_idx]

    F_sh_x = np.zeros(NN);  F_sh_y = np.zeros(NN)
    M_sh   = np.zeros(NN);  T_drv  = np.zeros(NN)
    JF_mtr = np.zeros(NN);  JF_lev = np.zeros(NN)
    JF_sl  = np.zeros(NN);  JF_cpl = np.zeros(NN)
    G_CONST = 9.81

    for _i, _h in enumerate(history):
        Ax, Ay = _h['Ax']*sc, _h['Ay']*sc
        Bx, By = _h['Bx']*sc, _h['By']*sc
        sy = _h['slider_y']*sc
        Px, Py = _h['Px']*sc, _h['Py']*sc
        beta = _h['beta']

        aAx, aAy = _h['A_Ax']*sc, _h['A_Ay']*sc
        aBx, aBy = _h['A_Bx']*sc, _h['A_By']*sc
        asy = _h['A_slider_y']*sc
        aPx, aPy = _h['A_Px']*sc, _h['A_Py']*sc

        vAx, vAy = _h['V_Ax']*sc, _h['V_Ay']*sc
        vBx, vBy = _h['V_Bx']*sc, _h['V_By']*sc
        vsy = _h['V_slider_y']*sc
        vPx, vPy = _h['V_Px']*sc, _h['V_Py']*sc

        a_CG_cr  = np.array([aAx/2, aAy/2])
        a_CG_lev = np.array([aBx/2, aBy/2])
        a_CG_tr  = np.array([aBx/2, (aBy + asy)/2])
        a_CG_cpl = np.array([aPx/2, aPy/2])

        CG_cr_p  = np.array([(Gc_m[0]+Ax)/2, (Gc_m[1]+Ay)/2])
        CG_lev_p = np.array([(Gl_m[0]+Bx)/2, (Gl_m[1]+By)/2])
        CG_tr_p  = np.array([Bx/2, (By+sy)/2])
        CG_cpl_p = np.array([(Xg_m+Px)/2, (Yg_m+Py)/2])

        v_CG_cr  = np.array([vAx/2, vAy/2])
        v_CG_lev = np.array([vBx/2, vBy/2])
        v_CG_tr  = np.array([vBx/2, (vBy+vsy)/2])
        v_CG_cpl = np.array([vPx/2, vPy/2])

        F_cr  = m_cr  * a_CG_cr
        F_lev = m_lev * a_CG_lev
        F_tr  = m_tr  * a_CG_tr
        F_sl  = np.array([0.0,  m_sl  * asy])
        F_obj = np.array([m_obj * aPx, m_obj * aPy])
        F_cpl = m_cpl * a_CG_cpl

        F_sh_x[_i] = F_cr[0]+F_lev[0]+F_tr[0]+F_sl[0]+F_obj[0]+F_cpl[0]
        F_sh_y[_i] = F_cr[1]+F_lev[1]+F_tr[1]+F_sl[1]+F_obj[1]+F_cpl[1]

        def _mom(r, F):
            return (r[0]-Gc_m[0])*F[1] - (r[1]-Gc_m[1])*F[0]

        M_sh[_i] = (_mom(CG_cr_p, F_cr) + _mom(CG_lev_p, F_lev) +
                    _mom(CG_tr_p, F_tr) + _mom(np.array([0.0, sy]), F_sl) +
                    _mom(np.array([Px, Py]), F_obj) + _mom(CG_cpl_p, F_cpl))

        _pwr = (np.dot(F_cr, v_CG_cr) + np.dot(F_lev, v_CG_lev) +
                np.dot(F_tr, v_CG_tr) + F_sl[1]*vsy +
                np.dot(F_obj, np.array([vPx, vPy])) + np.dot(F_cpl, v_CG_cpl))
        T_drv[_i] = _pwr / omega

        e_lev  = np.array([ math.cos(beta),  math.sin(beta)])
        e_perp = np.array([-math.sin(beta),  math.cos(beta)])

        JF_sl[_i]  = math.hypot(m_obj * aPx, (m_sl + m_obj) * (asy + G_CONST))

        cpl_vec    = np.array([Px - Xg_m, Py - Yg_m])
        cpl_len    = np.linalg.norm(cpl_vec) + 1e-12
        e_cpl      = cpl_vec / cpl_len
        JF_cpl[_i] = abs(np.dot(F_obj + F_cpl, e_cpl))

        L_lev_i     = max(math.hypot(Bx-Gl_m[0], By-Gl_m[1]), 1e-9)
        I_lev_Gl    = (1.0/3.0) * m_lev * L_lev_i**2
        alpha_lev_i = (aBx*e_perp[0] + aBy*e_perp[1]) / L_lev_i
        r_A_from_Gl = math.hypot(Ax-Gl_m[0], Ay-Gl_m[1])

        F_Bpin_y    = -(m_sl + m_obj) * (asy + G_CONST)
        F_Bpin      = np.array([0.0, F_Bpin_y])
        F_Bpin_perp = np.dot(F_Bpin, e_perp)

        N_A = ((I_lev_Gl * alpha_lev_i - F_Bpin_perp * L_lev_i) / max(r_A_from_Gl, 1e-9))
        F_A1  = N_A * e_perp
        F_Gl  = m_lev * a_CG_lev - F_A1 - F_Bpin
        JF_lev[_i] = np.linalg.norm(F_Gl)

        F_Gc  = m_cr * a_CG_cr + F_A1
        JF_mtr[_i] = np.linalg.norm(F_Gc)

    # ── GENERATE PLOTS ──
    _sh_mag = np.hypot(F_sh_x, F_sh_y)
    _mi_sh  = int(np.argmax(_sh_mag))

    fig1 = plt.figure(figsize=(16, 11))
    fig1.patch.set_facecolor('#f0f2f5')
    fig1.suptitle('Shaking Force · Shaking Moment · Driving Torque', fontsize=16, fontweight='bold')
    ax00 = fig1.add_subplot(2, 2, 1)
    ax01 = fig1.add_subplot(2, 2, 2)
    ax10 = fig1.add_subplot(2, 2, 3, projection='polar')
    ax11 = fig1.add_subplot(2, 2, 4)

    # Shaking Force Vector plot
    ax00.set_facecolor('#f8f9fa'); ax00.grid(True, color='white', lw=1.2)
    ax00.annotate('', xy=(F_sh_x[_mi_sh], F_sh_y[_mi_sh]), xytext=(0, 0), arrowprops=dict(arrowstyle='-|>', color='#c0392b', lw=2.8))
    ax00.plot(0, 0, 'ko', ms=8)
    _ext = max(abs(F_sh_x[_mi_sh]), abs(F_sh_y[_mi_sh]), 1e-9) * 1.7
    ax00.set_xlim(-_ext*0.15, _ext); ax00.set_ylim(-_ext*0.6, _ext*0.6)
    ax00.set_title(f'Inertial Shaking Force at φ = {phi_deg_all[_mi_sh]:.1f}°', fontweight='bold')
    ax00.set_xlabel('Force X [N]', fontweight='bold')
    ax00.set_ylabel('Force Y [N]', fontweight='bold')
    for spine in ax00.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')

    # Shaking Moment Plot
    _M_s = M_sh[sort_idx]
    _mi_M = int(np.argmax(np.abs(_M_s)))
    ax01.plot(phi_sorted, _M_s, color='#1565C0', lw=2.2)
    ax01.axhline(0, color='gray', lw=0.8, ls='--')
    ax01.plot(phi_sorted[_mi_M], _M_s[_mi_M], 'ro', ms=9)
    ax01.set_title('Shaking Moment vs Crank Angle', fontweight='bold')
    ax01.set_xlabel('Crank Angle [deg]', fontweight='bold')
    ax01.set_ylabel('Torque [N·m]', fontweight='bold')
    for spine in ax01.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')

    # Polar Plot
    _r_pol = np.abs(_M_s)
    _t_pol = np.deg2rad(phi_sorted)
    ax10.plot(_t_pol, _r_pol, color='#9b59b6', lw=2.2)
    ax10.fill(_t_pol, _r_pol, alpha=0.15, color='#9b59b6')
    ax10.plot(_t_pol[_mi_M], _r_pol[_mi_M], 'ro', ms=9)
    ax10.set_title('Polar Trace: |Shaking Moment| [N·m]', fontweight='bold', pad=18)
    # Polar spine formatting
    ax10.spines['polar'].set_linewidth(1.5)
    ax10.spines['polar'].set_color('black')

    # Driving Torque Plot
    _T_s = T_drv[sort_idx]
    _mi_T = int(np.argmax(np.abs(_T_s)))
    ax11.plot(phi_sorted, _T_s, color='black', lw=2.2)
    ax11.axhline(0, color='gray', lw=0.8, ls='--')
    ax11.plot(phi_sorted[_mi_T], _T_s[_mi_T], 'ro', ms=9)
    ax11.set_title('Required Motor Driving Torque', fontweight='bold')
    ax11.set_xlabel('Crank Angle [deg]', fontweight='bold')
    ax11.set_ylabel('Driving Torque [N·m]', fontweight='bold')
    for spine in ax11.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')

    # Add space between subplots
    fig1.tight_layout(pad=3.0, w_pad=2.5, h_pad=2.5)
    plt.show()

    # ── FIGURE 2: JOINT FORCES ──
    _jf_data = [
        (JF_mtr, '#1565C0', r'$F_{\mathrm{Motor\ Shaft}}$'),
        (JF_lev, '#2E7D32', r'$F_{\mathrm{Lever\ Pivot}}$'),
        (JF_sl,  '#c0392b', r'$F_{\mathrm{Slider\ Pin}}$'),
        (JF_cpl, '#AD1457', r'$F_{\mathrm{Coupler\ Pin}}$'),
    ]
    fig2, axs2 = plt.subplots(2, 2, figsize=(14, 10))
    fig2.patch.set_facecolor('#f0f2f5')
    fig2.suptitle('Joint Force vs Crank Angle  —  Aluminium, Newton-Euler', fontsize=16, fontweight='bold')

    for _ax, (_jf, _col, _ltx) in zip(axs2.flat, _jf_data):
        _jf_s = _jf[sort_idx]
        _jmax = int(np.argmax(_jf_s))
        _ax.plot(phi_sorted, _jf_s, color=_col, lw=2.2)
        _ax.plot(phi_sorted[_jmax], _jf_s[_jmax], 'ro', ms=8)
        _ax.grid(True, ls='--', alpha=0.5)
        _ax.set_title(f'Joint Force {_ltx} vs Crank Angle', fontweight='bold')
        _ax.set_xlabel('Crank Angle [deg]', fontweight='bold')
        _ax.set_ylabel('Force [N]', fontweight='bold')
        # Add thick borders to each subplot
        for spine in _ax.spines.values():
            spine.set_linewidth(1.5)
            spine.set_color('black')

    # Add space between subplots
    fig2.tight_layout(pad=3.0, w_pad=2.5, h_pad=2.5)
    plt.show()

    print("\n─── Peak Dynamic Quantities ─────────────────────────────────────")
    print(f"  Peak Driving Torque   : {np.max(np.abs(T_drv)):.5f} N·m  ")
    print(f"  Peak Shaking Force    : {np.max(_sh_mag):.5f} N  ")
    print(f"  Peak Shaking Moment   : {np.max(np.abs(M_sh)):.5f} N·m  ")
    print(f"  Peak Motor Shaft F    : {np.max(JF_mtr):.5f} N")
    print(f"  Peak Lever Pivot F    : {np.max(JF_lev):.5f} N")
    print(f"  Peak Slider Pin F     : {np.max(JF_sl):.5f} N")
    print(f"  Peak Coupler Pin F    : {np.max(JF_cpl):.5f} N")
    print("────────────────────────────────────────────────────────────────")

In [ ]:
#Block-11
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as PathEffects
from matplotlib.lines import Line2D

def generate_precision_snapshots(radius, drop_b, drop_c, density, omega, pin_u, pin_v, Xg, Yg, L_coupler):
    """
    Generates Velocity, Acceleration & Force Polygon Snapshots
    at Precision States A, B, and C.
    """
    print("\n" + "═"*75)
    print(" BLOCK 11: PRECISION STATE SNAPSHOTS (V, A, F Polygons) ")
    print("═"*75)

    A_CROSS_S = 1e-4    # m²  (10 mm × 10 mm bar cross-section)
    G_CONST   = 9.81    # m/s²

    # ── 1. GEOMETRY VALIDATION ──
    print("── Geometry Validation ──────────────────────────────────────────")
    if drop_b >= drop_c:
        raise ValueError(
            f"drop_b ({drop_b} mm) must be strictly less than drop_c ({drop_c} mm).\n"
            f"Synthesis is geometrically impossible: the three positions would be collinear.")

    _seg_frac = (drop_c - drop_b) / drop_c
    _rot_BC   = 90.0 * (1.0 - drop_b / drop_c)   # degrees of rotation in B→C

    if _seg_frac < 0.07:
        print(f"  WARNING — very short B→C segment detected!")
        print(f"   B→C slider travel  : {drop_c - drop_b:.2f} mm  ({_seg_frac*100:.1f}% of total stroke)")
        print(f"   B→C object rotation: {_rot_BC:.1f}°  (vs {90-_rot_BC:.1f}° for A→B)")
        print(f"   The object must rotate {_rot_BC:.1f}° while the slider moves only {drop_c-drop_b:.1f} mm.")
        print(f"   ➜ This creates a sharp kinematic 'jerk' near state C.")
        print(f"   ➜ Angular velocity & acceleration spike sharply in the last segment.")
        print(f"   ➜ Synthesis is mathematically valid; the jerk is physically real.\n")
    else:
        print(f"✓ Geometry OK  |  B→C = {drop_c-drop_b:.1f} mm  ({_seg_frac*100:.0f}% of stroke)  |  B→C rotation = {_rot_BC:.1f}°\n")

    # ── 2. SYNTHESIS & KINEMATICS (High Resolution) ──
    _eng_s = KinematicsEngine(drop_c, pin_u, pin_v, Xg, Yg, L_coupler, radius)
    _hist_s, _dt_s = compute_kinematics_history(_eng_s, omega, frames=300)

    # ── 3. LOCATE EXACT FRAMES FOR STATES A, B, C (BY DISPLACEMENT) ──
    _sy_s = np.array([h['slider_y'] for h in _hist_s])
    _theta_s = np.array([math.degrees(h['theta']) for h in _hist_s])

    _NS  = len(_hist_s)
    _iA  = int(np.argmax(_sy_s))
    _iC  = int(np.argmin(_sy_s))
    _fwd_s = (np.arange(_iA, _iC + 1) if _iC > _iA
              else np.concatenate([np.arange(_iA, _NS), np.arange(0, _iC + 1)]))

    # Base states from array
    h_A = dict(_hist_s[_iA])
    h_C = dict(_hist_s[_iC])

    # Target exact displacement for State B based on drop_b
    _tgt_sy_B = h_A['slider_y'] - drop_b

    # Safely interpolate to find exact displacement state without needing class methods
    idx_bfr = _fwd_s[0]
    idx_aft = _fwd_s[-1]

    for i in range(len(_fwd_s) - 1):
        y1 = _sy_s[_fwd_s[i]]
        y2 = _sy_s[_fwd_s[i+1]]
        if (y1 >= _tgt_sy_B and y2 <= _tgt_sy_B) or (y1 <= _tgt_sy_B and y2 >= _tgt_sy_B):
            idx_bfr = _fwd_s[i]
            idx_aft = _fwd_s[i+1]
            break

    y1 = _sy_s[idx_bfr]
    y2 = _sy_s[idx_aft]

    # Calculate exactly where the displacement falls between the two closest frames
    if abs(y2 - y1) < 1e-9:
        frac = 0.0
    else:
        frac = (_tgt_sy_B - y1) / (y2 - y1)

    h_B = {}
    for k, v in _hist_s[idx_bfr].items():
        if isinstance(v, (int, float, np.float64, np.float32)):
            # Linearly interpolate numeric data
            h_B[k] = v + frac * (_hist_s[idx_aft][k] - v)
        else:
            h_B[k] = v

    # Inject extra keys needed by draw_acceleration_polygon
    for h_tmp in [h_A, h_B, h_C]:
        h_tmp.update({
            'omega2' : omega,
            'alpha2' : 0.0,
            'r2'     : _eng_s.crank_r,
            'L_lever': _eng_s.L_lever_drive,
        })

    SNAP_STATES = [
        ('A', h_A, '#1565C0', f"{_iA}"),
        ('B', h_B, '#E65100', "Exact"),
        ('C', h_C, '#2E7D32', f"{_iC}"),
    ]

    # ── 4. GLOBAL SCALES ──
    _mv_all = max(math.hypot(h['V_Ax'], h['V_Ay']) for h in _hist_s)
    _ma_all = max(math.hypot(h['A_Ax'], h['A_Ay']) for h in _hist_s)
    SC_V_S  = 0.65 / (_mv_all + 1e-12)
    SC_A_S  = 0.65 / (_ma_all + 1e-12)

    _r_obj_m = radius * 1e-3
    _m_obj_s = density * A_CROSS_S * math.pi * _r_obj_m
    _I_obj_s = _m_obj_s * _r_obj_m**2 * ((3/8) - (8 /(27 * (math.pi**2))))

    # ── 5. PRINT STATE TABLE ──
    print(f"\n{'State':<6} {'φ (°)':<9} {'slider_y (mm)':<16} {'θ_obj (°)':<12} {'Frame'}")
    print("─" * 55)
    for _sid, _h, _, _lbl in SNAP_STATES:
        print(f"  {_sid}     {math.degrees(_h['phi'])%360:>7.1f}°   "
              f"{_h['slider_y']:>10.3f} mm    "
              f"{math.degrees(_h['theta']):>8.1f}°   [{_lbl}]")
    print()

    # ── 6. HELPER: FORCE POLYGON PANEL ──
    def _force_panel(ax, h, sid, scol):
        ax.cla(); ax.set_facecolor('#f8f9fa')
        ax.grid(True, color='white', lw=1.2)

        sc_mm = 1e-3
        _aPx = h['A_Px'] * sc_mm;  _aPy = h['A_Py'] * sc_mm

        F_in   = np.array([_m_obj_s * _aPx,     _m_obj_s * _aPy])
        F_grav = np.array([0.0,                 -_m_obj_s * G_CONST])
        F_drv  = -(F_in + F_grav)

        _F_list  = [F_in, F_grav, F_drv]
        _fc_list = ['#c0392b', '#27ae60', '#2980b9']
        _fl_list = ['Inertial  (F_in)', 'Weight  (F_grav)', 'Drive  (F_eq)']

        _fmax  = max(np.linalg.norm(f) for f in _F_list) or 1e-9
        _sf    = 0.5 / _fmax
        _all_x, _all_y = [0.0], [0.0]

        for _f, _fc, _fl in zip(_F_list, _fc_list, _fl_list):
            _ex, _ey = _f[0] * _sf, _f[1] * _sf
            ax.annotate('', xy=(_ex, _ey), xytext=(0, 0),
                        arrowprops=dict(arrowstyle='-|>', color=_fc, lw=2.5, mutation_scale=16))
            _t = ax.text((_ex)/2, (_ey)/2,
                         f'{_fl}\n{np.linalg.norm(_f)*1000:.2f} mN',
                         color=_fc, fontsize=8.5, fontweight='bold', va='center',
                         bbox=dict(fc='white', alpha=0.78, ec='none', pad=1.5))
            _t.set_path_effects([PathEffects.withStroke(linewidth=2, foreground='white')])
            _all_x.append(_ex); _all_y.append(_ey)

        ax.plot(0, 0, 'ko', ms=9, zorder=10)
        _span = max(max(_all_x) - min(_all_x), max(_all_y) - min(_all_y), 0.01)
        _mg   = _span * 0.45
        ax.set_xlim(min(_all_x) - _mg, max(_all_x) + _mg)
        ax.set_ylim(min(_all_y) - _mg, max(_all_y) + _mg)
        ax.set_aspect('equal')
        ax.set_title(f'Force Polygon on Object — State {sid}', fontsize=11, fontweight='bold', color=scol)
        ax.set_xlabel('Force X [N]'); ax.set_ylabel('Force Y [N]')
        ax.legend(handles=[
            Line2D([0],[0], color=c, lw=2, label=f'{l}  ({np.linalg.norm(f)*1000:.2f} mN)')
            for c, l, f in zip(_fc_list, _fl_list, _F_list)
        ], fontsize=8, loc='best', framealpha=0.9)

    # ── 7. GENERATE 3 SNAPSHOT FIGURES ──
    for _sid, _h, _scol, _ in SNAP_STATES:
        _phi_d = math.degrees(_h['phi']) % 360
        _th_d  = math.degrees(_h['theta'])

        fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
        fig.patch.set_facecolor('#e8eaf0')
        fig.suptitle(
            f'State  {_sid}  ·  φ = {_phi_d:.1f}°  ·  '
            f'slider_y = {_h["slider_y"]:.2f} mm  ·  θ_obj = {_th_d:.1f}°',
            fontsize=14, fontweight='bold', color=_scol)

        draw_velocity_polygon(axes[0], _h, scale=SC_V_S)
        axes[0].set_title(f'Velocity Polygon — State {_sid}  (φ = {_phi_d:.1f}°)',
                          fontsize=11, fontweight='bold', color=_scol)

        draw_acceleration_polygon(axes[1], _h, scale=SC_A_S)
        axes[1].set_title(f'Acceleration Polygon — State {_sid}  (φ = {_phi_d:.1f}°)',
                          fontsize=11, fontweight='bold', color=_scol)

        _force_panel(axes[2], _h, _sid, _scol)

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.show()



In [ ]:

#Block-9
from IPython.display import display, HTML
import math

class SynthAdapter:
    def __init__(self, drop_b, drop_c):
        self.drop_b = drop_b
        self.drop_c = drop_c
        self.states = [
            {'theta_deg': 0.0},
            {'theta_deg': 120.0},
            {'theta_deg': 240.0}
        ]

def main():
    print("=== Interactive Mechanism Generator ===")

    try:
        r_in = input("Enter Radius [Press Enter for default 50.0]: ")
        radius = float(r_in) if r_in.strip() else 50.0

        b_in = input("Enter Drop B [Press Enter for default 5.0]: ")
        drop_b = float(b_in) if b_in.strip() else 5.0

        c_in = input("Enter Drop C [Press Enter for default 15.0]: ")
        drop_c = float(c_in) if c_in.strip() else 15.0

        d_in = input("Enter Material Density in kg/m^3 [Press Enter for Aluminum: 2700]: ")
        density = float(d_in) if d_in.strip() else 2700.0

        e_in = input("Enter Elastic Modulus in MPa [Press Enter for Aluminum: 69000]: ")
        e_mod = float(e_in) if e_in.strip() else 69000.0

        sy_in = input("Enter Yield Strength in MPa [Press Enter for Aluminum: 276]: ")
        yield_s = float(sy_in) if sy_in.strip() else 276.0

        omega_in = input("Enter Crank Speed in rad/s [Press Enter for 10.0]: ")
        omega = float(omega_in) if omega_in.strip() else 10.0

    except ValueError:
        print("\nInvalid input detected. Falling back to default values.")
        radius, drop_b, drop_c, density, e_mod, yield_s, omega = 50.0, 5.0, 15.0, 2700.0, 69000.0, 276.0, 10.0

    print(f"\nExecuting with: Radius={radius}, Drop B={drop_b}, Drop C={drop_c}")
    print(f"Dynamics Setup: Density={density} kg/m^3, E={e_mod} MPa, Sy={yield_s} MPa, Omega={omega} rad/s")

    try:
        pin_u, pin_v, Xg, Yg, L_coupler = three_position_synthesis(radius, drop_b, drop_c)

        print(f"Success! Synthesized Ground Pivot: ({Xg:.1f}, {Yg:.1f})")
        print(f"Circle Pin Coordinates: u={pin_u:.1f}, v={pin_v:.1f}")

        print("\n--- 1/5: Generating Original Main Animation ---")
        animator = FullMechanismAnimator(
            drop_c=drop_c, u=pin_u, v=pin_v,
            Xg=Xg, Yg=Yg, L_coupler=L_coupler, radius=radius, omega=omega
        )
        anim_main = animator.create_animation()
        display(HTML(anim_main.to_jshtml()))

        print("\n--- 2/5: Generating Kinematic Polygons ---")
        anim_polygons = run_combined_animation(
            radius=radius, drop_c=drop_c, pin_u=pin_u, pin_v=pin_v,
            Xg=Xg, Yg=Yg, L_coupler=L_coupler, omega2=omega
        )
        display(HTML(anim_polygons.to_jshtml()))

        print("\n--- 3/5: Generating Static Dynamic Force Plots & Stress Test ---")
        calculated_mass = density * (0.1 * 0.1 * 0.05)
        calculated_inertia = 0.01
        synth_mock = SynthAdapter(drop_b=drop_b, drop_c=drop_c)

        text_analyzer = KinematicDynamicAnalyzer(
            synthesizer=synth_mock, mass=calculated_mass,
            inertia=calculated_inertia, omega_crank=omega
        )
        text_analyzer.perform_dynamic_analysis()
        text_analyzer.display_analysis()

        plot_engine = KinematicsEngine(drop_c, pin_u, pin_v, Xg, Yg, L_coupler, radius)
        history, dt = compute_kinematics_history(plot_engine, omega, frames=150)

        run_stress_test_and_dynamics(
            history=history,
            engine=plot_engine,
            density=density,
            E_mod=e_mod,
            Yield_S=yield_s,
            omega_crank=omega,
            radius=radius,
            L_coupler=L_coupler
        )

        print("\n--- 4/5: Generating Precision Snapshots (A, B, C) ---")
        # --- NEW: Generate the missing 9 snapshots natively ---
        generate_precision_snapshots(
            radius=radius, drop_b=drop_b, drop_c=drop_c,
            density=density, omega=omega,
            pin_u=pin_u, pin_v=pin_v, Xg=Xg, Yg=Yg, L_coupler=L_coupler
        )

        print("\n--- 5/5: Running Extended Modular Dynamic Analysis ---")
        run_extended_dynamic_analysis(
            radius=radius, drop_b=drop_b, drop_c=drop_c,
            density=density, omega=omega,
            pin_u=pin_u, pin_v=pin_v, Xg=Xg, Yg=Yg, L_coupler=L_coupler
        )

        print("\nAll generations completed successfully!")

    except Exception as e:
        import traceback
        print(f"\nSetup Error: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()

# **PLOT CODES FOR REFERENCE**